# Chapter 10: Combining Tables

The programs from Chapter 10 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## Merging on a Key

`merge_basics.py`

In [1]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")
products = pd.read_csv("../data/products.csv")
customers = pd.read_csv("../data/customers.csv")

full = orders.merge(
    products[["product", "unit_cost"]],
    on="product",
    how="left",
    validate="many_to_one",
).merge(
    customers[["customer_id", "segment"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)
full["revenue"] = (
    full["quantity"] * full["unit_price"] * (1 - full["discount"])
)
full["profit"] = full["revenue"] - full["quantity"] * full["unit_cost"]

print(full.shape)
summary = full.groupby("segment")[["revenue", "profit"]].sum()
summary["margin"] = summary["profit"] / summary["revenue"]
print(summary.round(2))

(5833, 14)
           revenue     profit  margin
segment                              
Business   77525.2   35084.24    0.45
Consumer  302178.2  135831.46    0.45


## Four Kinds of Join

`join_types.py`

In [2]:
import pandas as pd

left = pd.DataFrame({"id": [1, 2, 3], "name": ["Asha", "Ben", "Chen"]})
right = pd.DataFrame({"id": [2, 3, 4], "city": ["Leeds", "Pune", "Oslo"]})

for how in ["inner", "left", "right", "outer"]:
    print(f"--- {how}")
    print(left.merge(right, on="id", how=how, indicator=True))

--- inner
   id  name   city _merge
0   2   Ben  Leeds   both
1   3  Chen   Pune   both
--- left
   id  name   city     _merge
0   1  Asha    NaN  left_only
1   2   Ben  Leeds       both
2   3  Chen   Pune       both
--- right
   id  name   city      _merge
0   2   Ben  Leeds        both
1   3  Chen   Pune        both
2   4   NaN   Oslo  right_only
--- outer
   id  name   city      _merge
0   1  Asha    NaN   left_only
1   2   Ben  Leeds        both
2   3  Chen   Pune        both
3   4   NaN   Oslo  right_only


## Stacking Tables

`concat_rows.py`

In [3]:
import pandas as pd

q1 = pd.DataFrame({"store": ["A", "B"], "sales": [120, 95]})
q2 = pd.DataFrame({"store": ["A", "B"], "sales": [130, 90]})

both = pd.concat([q1, q2], keys=["Q1", "Q2"], names=["quarter", "row"])
print(both)
print(both.groupby("store")["sales"].sum())

            store  sales
quarter row             
Q1      0       A    120
        1       B     95
Q2      0       A    130
        1       B     90
store
A    250
B    185
Name: sales, dtype: int64
